# 社群網站 URL 初步比對分析器

本 Notebook 針對社群網站網址進行初步分析：

1. 建立常見社群網站官方網域資料庫。
2. 解析使用者輸入的 URL。
3. 比對 `http` / `https` 通訊協定。
4. 判斷網址是否屬於已知官方社群網站網域。
5. 偵測常見仿冒網域、Punycode、IP 網址與 `@` 混淆。
6. 輸出結構化分析結果與初步風險說明。

> HTTPS 只表示連線可能有加密，不代表網站一定安全。本 Notebook 是網址字串與網域比對模組，不取代黑名單、DNS、TLS、重新導向、網頁內容或惡意行為分析。

本版本不主動連線至輸入網址，因此可安全地先做離線字串比對。

## Cell 1：環境說明

本 Notebook 只使用 Python 標準函式庫與 `pandas`。Google Colab 通常已內建 `pandas`，不需額外安裝套件。

In [36]:
# 環境檢查
import sys

print(f"Python 版本：{sys.version.split()[0]}")
print("本 Notebook 不需額外下載網域解析套件。")

Python 版本：3.12.13
本 Notebook 不需額外下載網域解析套件。


## Cell 2：匯入套件與全域設定

In [37]:
from __future__ import annotations

import ipaddress
import json
import re
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from typing import Any
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
from IPython.display import display

# 常見多層 Public Suffix。
# 本專題的主要判斷仍以官方社群網域白名單為準。
MULTI_LABEL_SUFFIXES = {
    "com.tw", "org.tw", "net.tw", "edu.tw", "gov.tw",
    "co.uk", "org.uk", "ac.uk",
    "com.au", "net.au", "org.au",
    "co.jp", "ne.jp",
    "co.kr", "or.kr",
    "com.cn", "net.cn", "org.cn",
    "com.hk",
}

DATABASE_VERSION = "2026-07-23"

# 仿冒相似度門檻。門檻越低，召回率較高，但誤報也會增加。
LOOKALIKE_THRESHOLD = 0.72

# 分數只用於初步規則排序，不代表詐騙機率。
RISK_LEVELS = {
    "low": (0, 24),
    "medium": (25, 59),
    "high": (60, 100),
}

print("環境初始化完成。")

環境初始化完成。


## Cell 3：建立常見社群網站官方網域資料庫

資料庫採用「一個平台可對應多個官方網域」的設計。  
子網域不需要逐一列出，例如 `www.facebook.com`、`m.facebook.com` 都可由 `facebook.com` 判定。

此資料庫只是專題內部的可編輯白名單。

In [38]:
SOCIAL_MEDIA_DATABASE: list[dict[str, Any]] = [
    {
        "platform": "Facebook",
        "domains": ["facebook.com", "fb.com", "fb.me", "messenger.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["facebook", "fb", "messenger"],
    },
    {
        "platform": "Instagram",
        "domains": ["instagram.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["instagram", "insta"],
    },
    {
        "platform": "Threads",
        "domains": ["threads.net"],
        "expected_schemes": ["https"],
        "brand_tokens": ["threads"],
    },
    {
        "platform": "X / Twitter",
        "domains": ["x.com", "twitter.com", "t.co"],
        "expected_schemes": ["https"],
        "brand_tokens": ["twitter"],
    },
    {
        "platform": "TikTok",
        "domains": ["tiktok.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["tiktok"],
    },
    {
        "platform": "YouTube",
        "domains": ["youtube.com", "youtu.be"],
        "expected_schemes": ["https"],
        "brand_tokens": ["youtube", "youtu"],
    },
    {
        "platform": "LINE",
        "domains": ["line.me"],
        "expected_schemes": ["https"],
        "brand_tokens": ["line"],
    },
    {
        "platform": "WhatsApp",
        "domains": ["whatsapp.com", "wa.me"],
        "expected_schemes": ["https"],
        "brand_tokens": ["whatsapp"],
    },
    {
        "platform": "LinkedIn",
        "domains": ["linkedin.com", "lnkd.in"],
        "expected_schemes": ["https"],
        "brand_tokens": ["linkedin", "lnkd"],
    },
    {
        "platform": "Discord",
        "domains": ["discord.com", "discord.gg"],
        "expected_schemes": ["https"],
        "brand_tokens": ["discord"],
    },
    {
        "platform": "Telegram",
        "domains": ["telegram.org", "t.me"],
        "expected_schemes": ["https"],
        "brand_tokens": ["telegram"],
    },
    {
        "platform": "Reddit",
        "domains": ["reddit.com", "redd.it"],
        "expected_schemes": ["https"],
        "brand_tokens": ["reddit"],
    },
    {
        "platform": "Snapchat",
        "domains": ["snapchat.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["snapchat"],
    },
    {
        "platform": "Pinterest",
        "domains": ["pinterest.com", "pin.it"],
        "expected_schemes": ["https"],
        "brand_tokens": ["pinterest"],
    },
    {
        "platform": "Weibo",
        "domains": ["weibo.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["weibo"],
    },
    {
        "platform": "Dcard",
        "domains": ["dcard.tw"],
        "expected_schemes": ["https"],
        "brand_tokens": ["dcard"],
    },
    {
        "platform": "Plurk",
        "domains": ["plurk.com"],
        "expected_schemes": ["https"],
        "brand_tokens": ["plurk"],
    },
    {
        "platform": "Bluesky",
        "domains": ["bsky.app"],
        "expected_schemes": ["https"],
        "brand_tokens": ["bluesky", "bsky"],
    },
]


def build_domain_index(
    database: list[dict[str, Any]]
) -> dict[str, dict[str, Any]]:
    """將平台資料庫展開為 domain -> platform metadata 索引。"""
    index: dict[str, dict[str, Any]] = {}

    for item in database:
        for domain in item["domains"]:
            normalized_domain = domain.lower().strip(".")
            if normalized_domain in index:
                raise ValueError(f"重複的官方網域：{normalized_domain}")

            index[normalized_domain] = {
                "platform": item["platform"],
                "official_domain": normalized_domain,
                "expected_schemes": item["expected_schemes"],
                "brand_tokens": item["brand_tokens"],
            }

    return index


DOMAIN_INDEX = build_domain_index(SOCIAL_MEDIA_DATABASE)

print(f"已載入 {len(SOCIAL_MEDIA_DATABASE)} 個平台、{len(DOMAIN_INDEX)} 個官方網域。")

已載入 18 個平台、30 個官方網域。


## Cell 4：顯示與檢查社群網站資料庫

In [39]:
database_rows = []

for item in SOCIAL_MEDIA_DATABASE:
    database_rows.append(
        {
            "平台": item["platform"],
            "官方網域": ", ".join(item["domains"]),
            "預期協定": ", ".join(item["expected_schemes"]),
            "品牌關鍵字": ", ".join(item["brand_tokens"]),
        }
    )

database_df = pd.DataFrame(database_rows)
display(database_df)

print(f"資料庫版本：{DATABASE_VERSION}")

,平台,官方網域,預期協定,品牌關鍵字
0,Facebook,"facebook.com, fb.com, fb.me, messenger.com",https,"facebook, fb, messenger"
1,Instagram,instagram.com,https,"instagram, insta"
2,Threads,threads.net,https,threads
3,X / Twitter,"x.com, twitter.com, t.co",https,twitter
4,TikTok,tiktok.com,https,tiktok
5,YouTube,"youtube.com, youtu.be",https,"youtube, youtu"
6,LINE,line.me,https,line
7,WhatsApp,"whatsapp.com, wa.me",https,whatsapp
8,LinkedIn,"linkedin.com, lnkd.in",https,"linkedin, lnkd"
9,Discord,"discord.com, discord.gg",https,discord


資料庫版本：2026-07-23


## Cell 5：定義輸出資料結構

使用 dataclass 固定每次分析的輸出欄位，方便後續串接前端、風險整合模組或 LLM 報告。

In [40]:
@dataclass
class ParsedURL:
    original_url: str
    parse_url: str
    normalized_url: str
    supplied_scheme: str | None
    effective_scheme: str
    scheme_was_missing: bool
    hostname: str
    unicode_hostname: str
    port: int | None
    path: str
    query: str
    registered_domain: str
    subdomain: str
    suffix: str
    has_punycode: bool
    uses_ip_address: bool
    contains_at_symbol: bool


@dataclass
class URLAnalysisResult:
    original_url: str
    normalized_url: str
    hostname: str
    registered_domain: str
    supplied_scheme: str | None
    effective_scheme: str
    scheme_was_missing: bool
    protocol_status: str
    is_known_social_domain: bool
    matched_platform: str | None
    matched_official_domain: str | None
    domain_match_type: str
    possible_impersonated_platform: str | None
    possible_target_domain: str | None
    lookalike_similarity: float
    has_punycode: bool
    uses_ip_address: bool
    contains_at_symbol: bool
    risk_score: int
    risk_level: str
    evidence: list[str]


## Cell 6：URL 正規化與解析

- 沒有輸入協定時，暫時以 `https://` 解析，但保留「原始網址缺少協定」資訊。
- 只接受 `http` 與 `https`。
- 使用 `urlsplit()` 解析真正 hostname，避免 `https://facebook.com@evil.com` 被誤判成 Facebook。
- 同時保留 ASCII/Punycode 與 Unicode hostname。

In [41]:
SUPPORTED_SCHEMES = {"http", "https"}


def is_ip_hostname(hostname: str) -> bool:
    """判斷 hostname 是否為 IPv4 或 IPv6。"""
    try:
        ipaddress.ip_address(hostname.strip("[]"))
        return True
    except ValueError:
        return False


def decode_idna_hostname(hostname: str) -> str:
    """盡可能將 Punycode hostname 解碼為 Unicode，失敗時回傳原值。"""
    try:
        return hostname.encode("ascii").decode("idna")
    except (UnicodeError, UnicodeEncodeError):
        return hostname


def split_domain_parts(hostname: str) -> tuple[str, str, str]:
    """
    回傳 (registered_domain, subdomain, suffix)。

    這是針對專題白名單比對的輕量解析器，不是完整 Public Suffix List 實作。
    """
    hostname = hostname.lower().strip(".")
    labels = [label for label in hostname.split(".") if label]

    if len(labels) == 1:
        return hostname, "", ""

    last_two = ".".join(labels[-2:])

    if last_two in MULTI_LABEL_SUFFIXES and len(labels) >= 3:
        suffix = last_two
        registered_domain = ".".join(labels[-3:])
        subdomain = ".".join(labels[:-3])
    else:
        suffix = labels[-1]
        registered_domain = ".".join(labels[-2:])
        subdomain = ".".join(labels[:-2])

    return registered_domain, subdomain, suffix


def parse_and_normalize_url(raw_url: str) -> ParsedURL:
    """驗證、正規化並拆解使用者輸入的 URL。"""
    if not isinstance(raw_url, str):
        raise TypeError("URL 必須是字串。")

    original_url = raw_url.strip()
    if not original_url:
        raise ValueError("URL 不可為空白。")

    explicit_scheme_match = re.match(
        r"^([A-Za-z][A-Za-z0-9+.-]*)://",
        original_url,
    )
    supplied_scheme = (
        explicit_scheme_match.group(1).lower()
        if explicit_scheme_match
        else None
    )
    scheme_was_missing = supplied_scheme is None

    if supplied_scheme and supplied_scheme not in SUPPORTED_SCHEMES:
        raise ValueError(
            f"不支援的協定：{supplied_scheme}。本分析器只接受 http 或 https。"
        )

    parse_url = original_url if supplied_scheme else f"https://{original_url}"
    parts = urlsplit(parse_url)

    effective_scheme = parts.scheme.lower()
    if effective_scheme not in SUPPORTED_SCHEMES:
        raise ValueError("URL 必須使用 http 或 https。")

    if not parts.hostname:
        raise ValueError("無法解析 hostname，請確認 URL 格式。")

    hostname = parts.hostname.lower().rstrip(".")

    try:
        ascii_hostname = hostname.encode("idna").decode("ascii").lower()
    except UnicodeError as exc:
        raise ValueError("hostname 含有無法處理的 Unicode 字元。") from exc

    unicode_hostname = decode_idna_hostname(ascii_hostname)
    uses_ip_address = is_ip_hostname(ascii_hostname)

    if uses_ip_address:
        registered_domain = ascii_hostname
        subdomain = ""
        suffix = ""
    else:
        registered_domain, subdomain, suffix = split_domain_parts(ascii_hostname)

    normalized_netloc = ascii_hostname
    if parts.port is not None:
        normalized_netloc = f"{ascii_hostname}:{parts.port}"

    # 移除 fragment，但保留 path 與 query。
    normalized_url = urlunsplit(
        (
            effective_scheme,
            normalized_netloc,
            parts.path or "/",
            parts.query,
            "",
        )
    )

    return ParsedURL(
        original_url=original_url,
        parse_url=parse_url,
        normalized_url=normalized_url,
        supplied_scheme=supplied_scheme,
        effective_scheme=effective_scheme,
        scheme_was_missing=scheme_was_missing,
        hostname=ascii_hostname,
        unicode_hostname=unicode_hostname,
        port=parts.port,
        path=parts.path or "/",
        query=parts.query,
        registered_domain=registered_domain,
        subdomain=subdomain,
        suffix=suffix,
        has_punycode="xn--" in ascii_hostname,
        uses_ip_address=uses_ip_address,
        contains_at_symbol="@" in original_url,
    )

## Cell 7：官方網域比對

合法匹配條件只有兩種：

1. hostname 與官方網域完全相同。
2. hostname 是官方網域的真正子網域。

因此 `facebook.com.evil.com` 不會被判定為 `facebook.com`。

In [42]:
def hostname_matches_domain(hostname: str, official_domain: str) -> bool:
    """判斷 hostname 是否等於官方網域或其真正子網域。"""
    return hostname == official_domain or hostname.endswith(f".{official_domain}")


def match_official_social_domain(
    hostname: str,
) -> dict[str, Any] | None:
    """比對已知官方社群網站網域，優先採用最長網域匹配。"""
    for official_domain in sorted(DOMAIN_INDEX, key=len, reverse=True):
        if hostname_matches_domain(hostname, official_domain):
            metadata = DOMAIN_INDEX[official_domain].copy()
            metadata["match_type"] = (
                "exact_domain"
                if hostname == official_domain
                else "official_subdomain"
            )
            return metadata

    return None

## Cell 8：仿冒網域與品牌相似度分析

此功能只產生候選警示，不直接證明詐騙。  
例如比較 `faceb00k-login.com` 與 `facebook.com`。

過短品牌（如 `x.com`）不執行模糊相似度，避免大量誤報。

In [43]:
LEETSPEAK_TRANSLATION = str.maketrans({
    "0": "o",
    "1": "i",
    "3": "e",
    "4": "a",
    "5": "s",
    "7": "t",
})


def domain_label(domain: str) -> str:
    """取得註冊網域主要標籤，例如 facebook.com -> facebook。"""
    registered_domain, _, suffix = split_domain_parts(domain)

    if suffix and registered_domain.endswith(f".{suffix}"):
        return registered_domain[: -(len(suffix) + 1)].split(".")[-1]

    return registered_domain.split(".")[0]


def canonicalize_label(label: str) -> str:
    """統一大小寫、常見 leetspeak，並移除非英數字元。"""
    normalized = label.lower().translate(LEETSPEAK_TRANSLATION)
    return re.sub(r"[^a-z0-9]", "", normalized)


def sequence_similarity(left: str, right: str) -> float:
    """計算兩字串的 SequenceMatcher 相似度。"""
    if not left or not right:
        return 0.0
    return SequenceMatcher(None, left, right).ratio()


def find_possible_impersonation(
    parsed: ParsedURL,
) -> dict[str, Any]:
    """找出最可能被模仿的平台與官方網域。"""
    empty_result = {
        "platform": None,
        "official_domain": None,
        "similarity": 0.0,
        "brand_token_found": False,
    }

    if parsed.uses_ip_address:
        return empty_result

    unknown_label = canonicalize_label(domain_label(parsed.registered_domain))
    hostname_compact = canonicalize_label(parsed.hostname)
    best_candidate = empty_result.copy()

    for official_domain, metadata in DOMAIN_INDEX.items():
        official_label = canonicalize_label(domain_label(official_domain))

        # 過短標籤不適合模糊比較，避免 x.com、t.me 等造成大量誤報。
        if len(official_label) < 4:
            similarity = 0.0
        else:
            similarity = sequence_similarity(unknown_label, official_label)

        brand_token_found = any(
            len(canonicalize_label(token)) >= 4
            and canonicalize_label(token) in hostname_compact
            for token in metadata["brand_tokens"]
        )

        candidate_score = max(similarity, 0.80 if brand_token_found else 0.0)

        if candidate_score > best_candidate["similarity"]:
            best_candidate = {
                "platform": metadata["platform"],
                "official_domain": official_domain,
                "similarity": round(candidate_score, 4),
                "brand_token_found": brand_token_found,
            }

    # 沒達門檻就不要顯示一個容易誤導的「可能仿冒平台」。
    if best_candidate["similarity"] < LOOKALIKE_THRESHOLD:
        return empty_result

    return best_candidate

## Cell 9：HTTP / HTTPS 協定比對

- 已知官方社群網域且使用 HTTPS：符合資料庫預期。
- 已知官方社群網域但使用 HTTP：網域匹配，但傳輸未加密。
- 未提供協定：分析器暫以 HTTPS 解析，但不能聲稱原始輸入包含 HTTPS。
- 陌生網域：只描述協定本身，不把 HTTPS 當作可信證據。

In [44]:
def evaluate_protocol(
    parsed: ParsedURL,
    official_match: dict[str, Any] | None,
) -> str:
    """評估使用者輸入的 http/https 狀態。"""
    if parsed.scheme_was_missing:
        return "scheme_missing_assumed_https_for_parsing"

    if official_match is not None:
        if parsed.effective_scheme in official_match["expected_schemes"]:
            return "matches_expected_https"
        return "official_domain_but_insecure_http"

    if parsed.effective_scheme == "https":
        return "unknown_domain_using_https"

    return "unknown_domain_using_http"

## Cell 10：規則式初步風險評分

分數用途是排序與提示，不是機率，也不是已訓練完成的詐騙分類器。

In [45]:
def score_to_level(score: int) -> str:
    if score <= RISK_LEVELS["low"][1]:
        return "low"
    if score <= RISK_LEVELS["medium"][1]:
        return "medium"
    return "high"


def calculate_initial_risk(
    parsed: ParsedURL,
    official_match: dict[str, Any] | None,
    impersonation: dict[str, Any],
    protocol_status: str,
) -> tuple[int, list[str]]:
    """根據可解釋規則計算初步風險分數與證據。"""
    score = 0
    evidence: list[str] = []

    if official_match is not None:
        evidence.append(
            f"hostname 符合 {official_match['platform']} 的官方網域 "
            f"{official_match['official_domain']}。"
        )
    else:
        score += 30
        evidence.append("hostname 不在目前的社群網站官方網域資料庫中，需進一步查證。")

    if protocol_status == "matches_expected_https":
        evidence.append("輸入網址明確使用 HTTPS，符合資料庫預期。")
    elif protocol_status == "official_domain_but_insecure_http":
        score += 40
        evidence.append("網址雖符合官方網域，但使用 HTTP；傳輸內容可能未加密。")
    elif protocol_status == "scheme_missing_assumed_https_for_parsing":
        score += 10
        evidence.append("原始輸入未提供 http/https；系統僅為解析而暫時假設 HTTPS。")
    elif protocol_status == "unknown_domain_using_https":
        evidence.append("陌生網域使用 HTTPS；這只表示連線可能加密，不代表網站可信。")
    elif protocol_status == "unknown_domain_using_http":
        score += 25
        evidence.append("陌生網域使用 HTTP，且不在官方社群網域資料庫中。")

    if official_match is None:
        similarity = impersonation["similarity"]
        if similarity >= LOOKALIKE_THRESHOLD:
            score += 40
            evidence.append(
                f"網域可能模仿 {impersonation['platform']}；"
                f"最接近官方網域 {impersonation['official_domain']}，"
                f"相似度為 {similarity:.2f}。"
            )

    if parsed.has_punycode:
        score += 20
        evidence.append("hostname 包含 Punycode（xn--），需留意 Unicode 同形字混淆。")

    if parsed.uses_ip_address:
        score += 25
        evidence.append("網址直接使用 IP 位址，而非一般註冊網域。")

    if parsed.contains_at_symbol:
        score += 35
        evidence.append("原始 URL 含有 @；瀏覽器實際連線主機應以 @ 後方 hostname 為準。")

    if parsed.port not in (None, 80, 443):
        score += 10
        evidence.append(f"網址使用非標準連接埠 {parsed.port}。")

    return min(score, 100), evidence

## Cell 11：整合主分析函式

In [46]:
def analyze_social_media_url(raw_url: str) -> URLAnalysisResult:
    """執行社群網站 URL 的完整初步比對分析。"""
    parsed = parse_and_normalize_url(raw_url)
    official_match = match_official_social_domain(parsed.hostname)

    if official_match is None:
        impersonation = find_possible_impersonation(parsed)
    else:
        impersonation = {
            "platform": None,
            "official_domain": None,
            "similarity": 0.0,
            "brand_token_found": False,
        }

    protocol_status = evaluate_protocol(parsed, official_match)

    risk_score, evidence = calculate_initial_risk(
        parsed=parsed,
        official_match=official_match,
        impersonation=impersonation,
        protocol_status=protocol_status,
    )


    return URLAnalysisResult(
        original_url=parsed.original_url,
        normalized_url=parsed.normalized_url,
        hostname=parsed.hostname,
        registered_domain=parsed.registered_domain,
        supplied_scheme=parsed.supplied_scheme,
        effective_scheme=parsed.effective_scheme,
        scheme_was_missing=parsed.scheme_was_missing,
        protocol_status=protocol_status,
        is_known_social_domain=official_match is not None,
        matched_platform=official_match["platform"] if official_match else None,
        matched_official_domain=official_match["official_domain"] if official_match else None,
        domain_match_type=official_match["match_type"] if official_match else "no_official_match",
        possible_impersonated_platform=impersonation["platform"],
        possible_target_domain=impersonation["official_domain"],
        lookalike_similarity=float(impersonation["similarity"]),
        has_punycode=parsed.has_punycode,
        uses_ip_address=parsed.uses_ip_address,
        contains_at_symbol=parsed.contains_at_symbol,
        risk_score=risk_score,
        risk_level=score_to_level(risk_score),
        evidence=evidence
    )

## Cell 12：結果顯示函式

同時提供摘要表格、判斷證據、限制說明與可供 API/LLM 使用的 JSON。

In [47]:
PROTOCOL_STATUS_ZH = {
    "matches_expected_https": "官方網域且明確使用 HTTPS",
    "official_domain_but_insecure_http": "官方網域但使用 HTTP",
    "scheme_missing_assumed_https_for_parsing": "原始輸入缺少協定",
    "unknown_domain_using_https": "陌生網域使用 HTTPS",
    "unknown_domain_using_http": "陌生網域使用 HTTP",
}

RISK_LEVEL_ZH = {
    "low": "低",
    "medium": "中",
    "high": "高",
}


def display_analysis(result: URLAnalysisResult) -> None:
    """以表格、證據與 JSON 顯示分析結果。"""
    summary = pd.DataFrame(
        [
            {"欄位": "原始輸入", "結果": result.original_url},
            {"欄位": "正規化 URL", "結果": result.normalized_url},
            {"欄位": "Hostname", "結果": result.hostname},
            {"欄位": "註冊網域", "結果": result.registered_domain},
            {"欄位": "使用者輸入協定", "結果": result.supplied_scheme or "未提供"},
            {"欄位": "協定判斷", "結果": PROTOCOL_STATUS_ZH[result.protocol_status]},
            {
                "欄位": "是否為已知官方社群網域",
                "結果": "是" if result.is_known_social_domain else "否",
            },
            {"欄位": "匹配平台", "結果": result.matched_platform or "無"},
            {"欄位": "匹配官方網域", "結果": result.matched_official_domain or "無"},
            {
                "欄位": "可能仿冒平台",
                "結果": result.possible_impersonated_platform or "未發現明顯候選",
            },
            {"欄位": "相似度", "結果": f"{result.lookalike_similarity:.2f}"},
            {"欄位": "初步風險分數", "結果": f"{result.risk_score}/100"},
            {"欄位": "初步風險等級", "結果": RISK_LEVEL_ZH[result.risk_level]},
        ]
    )

    display(summary)

    print("\n【判斷證據】")
    for index, item in enumerate(result.evidence, start=1):
        print(f"{index}. {item}")


    print("\n【JSON 輸出】")
    print(json.dumps(asdict(result), ensure_ascii=False, indent=2))

## Cell 14：輸入單一 URL 進行分析

執行此 Cell 後，在輸入框貼上網址。

In [48]:
user_url = input("請輸入要分析的 URL：").strip()

try:
    analysis_result = analyze_social_media_url(user_url)
    display_analysis(analysis_result)
except (TypeError, ValueError) as exc:
    print(f"輸入錯誤：{exc}")
except Exception as exc:
    print(f"分析時發生未預期錯誤：{type(exc).__name__}: {exc}")

請輸入要分析的 URL：http://youtube.com


,欄位,結果
0,原始輸入,http://youtube.com
1,正規化 URL,http://youtube.com/
2,Hostname,youtube.com
3,註冊網域,youtube.com
4,使用者輸入協定,http
5,協定判斷,官方網域但使用 HTTP
6,是否為已知官方社群網域,是
7,匹配平台,YouTube
8,匹配官方網域,youtube.com
9,可能仿冒平台,未發現明顯候選



【判斷證據】
1. hostname 符合 YouTube 的官方網域 youtube.com。
2. 網址雖符合官方網域，但使用 HTTP；傳輸內容可能未加密。

【JSON 輸出】
{
  "original_url": "http://youtube.com",
  "normalized_url": "http://youtube.com/",
  "hostname": "youtube.com",
  "registered_domain": "youtube.com",
  "supplied_scheme": "http",
  "effective_scheme": "http",
  "scheme_was_missing": false,
  "protocol_status": "official_domain_but_insecure_http",
  "is_known_social_domain": true,
  "matched_platform": "YouTube",
  "matched_official_domain": "youtube.com",
  "domain_match_type": "exact_domain",
  "possible_impersonated_platform": null,
  "possible_target_domain": null,
  "lookalike_similarity": 0.0,
  "has_punycode": false,
  "uses_ip_address": false,
  "contains_at_symbol": false,
  "risk_score": 40,
  "risk_level": "medium",
  "evidence": [
    "hostname 符合 YouTube 的官方網域 youtube.com。",
    "網址雖符合官方網域，但使用 HTTP；傳輸內容可能未加密。"
  ]
}


## Cell 15：以變數方式重複測試

需要反覆測試時，只修改 `USER_URL`。

In [49]:
USER_URL = "http://youtube.com"

result = analyze_social_media_url(USER_URL)
display_analysis(result)

,欄位,結果
0,原始輸入,http://youtube.com
1,正規化 URL,http://youtube.com/
2,Hostname,youtube.com
3,註冊網域,youtube.com
4,使用者輸入協定,http
5,協定判斷,官方網域但使用 HTTP
6,是否為已知官方社群網域,是
7,匹配平台,YouTube
8,匹配官方網域,youtube.com
9,可能仿冒平台,未發現明顯候選



【判斷證據】
1. hostname 符合 YouTube 的官方網域 youtube.com。
2. 網址雖符合官方網域，但使用 HTTP；傳輸內容可能未加密。

【JSON 輸出】
{
  "original_url": "http://youtube.com",
  "normalized_url": "http://youtube.com/",
  "hostname": "youtube.com",
  "registered_domain": "youtube.com",
  "supplied_scheme": "http",
  "effective_scheme": "http",
  "scheme_was_missing": false,
  "protocol_status": "official_domain_but_insecure_http",
  "is_known_social_domain": true,
  "matched_platform": "YouTube",
  "matched_official_domain": "youtube.com",
  "domain_match_type": "exact_domain",
  "possible_impersonated_platform": null,
  "possible_target_domain": null,
  "lookalike_similarity": 0.0,
  "has_punycode": false,
  "uses_ip_address": false,
  "contains_at_symbol": false,
  "risk_score": 40,
  "risk_level": "medium",
  "evidence": [
    "hostname 符合 YouTube 的官方網域 youtube.com。",
    "網址雖符合官方網域，但使用 HTTP；傳輸內容可能未加密。"
  ]
}


## Cell 16：新增或修改平台資料庫範例

修改 `SOCIAL_MEDIA_DATABASE` 後，必須重新建立 `DOMAIN_INDEX`。  
正式系統建議把資料庫移至 JSON、CSV 或資料庫，並記錄版本與更新日期。

In [50]:
# 範例：
#
# SOCIAL_MEDIA_DATABASE.append(
#     {
#         "platform": "ExampleSocial",
#         "domains": ["example-social.com", "ex.social"],
#         "expected_schemes": ["https"],
#         "brand_tokens": ["examplesocial"],
#     }
# )
#
# DOMAIN_INDEX = build_domain_index(SOCIAL_MEDIA_DATABASE)
# print(f"目前共有 {len(DOMAIN_INDEX)} 個官方網域。")